# 04 · 期权定价入门：Black-Scholes、希腊字母与蒙特卡洛

**学习目标**：
1. 用 Black-Scholes 公式给欧式期权定价，并验证看涨-看跌平价关系；
2. 理解希腊字母：Delta、Gamma、Vega、Theta 分别衡量什么风险；
3. 用蒙特卡洛模拟给同一个期权定价，看它如何随模拟次数收敛到解析解。

只依赖 numpy / pandas / matplotlib，**无需联网**。

> 本仓库仅供学习，所有内容不构成投资建议。期权是高风险工具，真实交易前务必了解保证金、行权和流动性规则。

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.4f}".format)


def norm_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))


def norm_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2 * math.pi)

## 1. Black-Scholes 公式

欧式看涨期权价格 `C = S·N(d1) − K·e^(−rT)·N(d2)`，看跌期权 `P = K·e^(−rT)·N(−d2) − S·N(−d1)`，其中

- `d1 = [ln(S/K) + (r + σ²/2)·T] / (σ·√T)`，`d2 = d1 − σ·√T`
- S 标的价格，K 行权价，T 到期年数，r 无风险利率，σ 年化波动率

模型假设：价格服从对数正态分布、波动率和利率不变、无交易成本、可以连续对冲。真实市场都不完全满足，这是理解“隐含波动率微笑”等现象的起点。

In [ ]:
def bs_price(S, K, T, r, sigma, kind="call"):
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if kind == "call":
        return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)


S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.03, 0.20
call = bs_price(S, K, T, r, sigma, "call")
put = bs_price(S, K, T, r, sigma, "put")
print(f"看涨期权价格: {call:.4f}")
print(f"看跌期权价格: {put:.4f}")

## 2. 看涨-看跌平价

无套利条件下 `C − P = S − K·e^(−rT)`。两边相差明显时理论上存在套利机会，所以它常被用来检查报价或代码有没有错。

In [ ]:
lhs = call - put
rhs = S - K * math.exp(-r * T)
print(f"C - P          = {lhs:.6f}")
print(f"S - K*e^(-rT)  = {rhs:.6f}")
assert abs(lhs - rhs) < 1e-9

## 3. 希腊字母

| 指标 | 含义 |
| --- | --- |
| Delta | 标的涨 1 元，期权价格变动多少；也是对冲需要持有的标的数量 |
| Gamma | Delta 本身随标的价格变化的速度，衡量对冲需要多频繁地调整 |
| Vega | 波动率上升 1 个百分点，期权价格变动多少 |
| Theta | 时间每过一天，期权价格损失多少（时间价值衰减） |

In [ ]:
def bs_greeks_call(S, K, T, r, sigma):
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    delta = norm_cdf(d1)
    gamma = norm_pdf(d1) / (S * sigma * math.sqrt(T))
    vega = S * norm_pdf(d1) * math.sqrt(T) / 100          # 每 1 个百分点波动率
    theta = (-S * norm_pdf(d1) * sigma / (2 * math.sqrt(T))
             - r * K * math.exp(-r * T) * norm_cdf(d2)) / 365   # 每天
    return {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta}


spots = np.linspace(70, 130, 61)
greeks = pd.DataFrame([bs_greeks_call(s, K, T, r, sigma) for s in spots], index=spots)
greeks.loc[[80.0, 100.0, 120.0]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
greeks["delta"].plot(ax=axes[0], title="Call delta vs spot")
greeks["gamma"].plot(ax=axes[1], title="Call gamma vs spot")
for ax in axes:
    ax.axvline(K, color="grey", linestyle="--")
    ax.set_xlabel("spot price")
plt.tight_layout()
plt.show()

可以看到：深度实值时 Delta 接近 1，深度虚值时接近 0；Gamma 在平值附近最大，说明平值期权的对冲最需要频繁调整。

## 4. 蒙特卡洛定价

在风险中性假设下模拟到期价格 `S_T = S·exp((r − σ²/2)T + σ√T·Z)`，对到期收益取平均再折现。模拟次数越多，结果越接近解析解，误差大约按 `1/√N` 缩小。

In [ ]:
rng = np.random.default_rng(2026)


def mc_call(S, K, T, r, sigma, n):
    z = rng.standard_normal(n)
    st = S * np.exp((r - 0.5 * sigma ** 2) * T + sigma * math.sqrt(T) * z)
    payoff = np.maximum(st - K, 0.0)
    price = math.exp(-r * T) * payoff.mean()
    stderr = math.exp(-r * T) * payoff.std(ddof=1) / math.sqrt(n)
    return price, stderr


rows = []
for n in [1_000, 10_000, 100_000, 1_000_000]:
    price, se = mc_call(S, K, T, r, sigma, n)
    rows.append({"paths": n, "mc_price": price, "std_error": se, "abs_error_vs_bs": abs(price - call)})
mc = pd.DataFrame(rows).set_index("paths")
mc

## 练习

1. 把波动率从 20% 改成 40%，看涨和看跌期权价格分别怎么变？用 Vega 解释。
2. 给定市场上的期权价格，用二分法反解出**隐含波动率**。
3. 用蒙特卡洛给一个**亚式期权**（按到期前每日均价结算）定价。它没有简单的解析公式，这正是蒙特卡洛的用武之地。
4. 对照 MIT 18.S096（见 TOP_RESOURCES）中关于 Black-Scholes 推导的讲义，理解“风险中性定价”的含义。